# Entorno

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import torch
torch.__version__
torch.cuda.is_available()

# Dataset

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", None)

df = pd.read_csv("https://raw.githubusercontent.com/eduardofc/data/refs/heads/main/opl_programming_language_2.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.sample(5, random_state=99)

In [ ]:
print(df.iloc[38]['response'])

# Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

# Before fine-tuning

In [ ]:
FastLanguageModel.for_inference(model)

In [ ]:
template = """Below are some instructions that describe some tasks. Write responses that appropriately complete each request.

### Instruction:
{INPUT}

### Response:
{OUTPUT}"""

In [ ]:
QUERY = "Write a hello world program in the OPL programming language."

In [ ]:
template.format(INPUT=QUERY, OUTPUT="")

In [ ]:
inputs = tokenizer([template.format(INPUT=QUERY, OUTPUT="")], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])

In [ ]:
tokenizer.eos_token

# Fine-tuning

In [ ]:
from datasets import Dataset

ds = Dataset.from_pandas(df)
ds

In [ ]:
ds[38]

In [ ]:
def formatting_prompts(x):
  input = x['question']
  output = x['response']
  x['text'] = template.format(INPUT=input, OUTPUT=output) + tokenizer.eos_token
  return x

ds = ds.map(formatting_prompts).remove_columns(['question', 'response'])
ds

In [ ]:
ds[38]

In [ ]:
from trl import SFTConfig, SFTTrainer

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [ ]:
trainer.train()

# After fine-tuning

In [ ]:
inputs = tokenizer([template.format(INPUT=QUERY, OUTPUT="")], return_tensors='pt').to('cuda')
outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True)
print(tokenizer.batch_decode(outputs)[0])